# Conceptual Pre-requisites for `Canonicalizing Open Knowledge Bases` Paper

## 1. Knowledge Bases

### What Is A Knowledge Base?
In layman terms, a `Knowledge Base` is a structured way to represent facts about the world.

Imagine on one hand we have a book which has facts and sentences written in the random way and every knowledge is scattered. On the other hand we have a notebook where we write down the facts like:
- "Barack Obama was born in Honolulu"
- "Apple Inc. was founded by Steve Jobs"

Instead of random sentences we write these facts in a machine-readable form:
```text
(Barack Obama, place of birth, Honolulu)
(Apple Inc., founded by, Steve Jobs)
```

Here each fact is called a triple: `(subject, predicate, object)`.


### Core Components of a KB

#### 1. Triples (aka Facts)

Form:

```
(subject, predicate, object)  
= (s, p, o)
```

Example:

```
("Paris", "isCapitalOf", "France")
```

Each triple asserts a relationship.

#### 2. Entities
* **Subject/Object**: real-world items (people, places, things)
* Stored with unique IDs, e.g., Freebase ID `/m/06cx9` for Barack Obama

#### 3. Relations / Predicates
* The link between subject and object, e.g., `bornIn`, `hasChild`, `presidentOf`



### Types of Knowledge Bases
| Type             | Description                                        | Example           |
| ---------------- | -------------------------------------------------- | ----------------- |
| Open KB          | Facts are extracted from text, no strict schema    | ReVerb, OpenIE    |
| Closed KB        | Schema-defined with predefined relations and types | Freebase, DBpedia |
| Probabilistic KB | Stores facts with uncertainty/confidence           | Knowledge Vault   |


### Formal Foundation

####  KB as a Set

Let:

* 𝔼 = set of entities
* ℛ = set of relations
* 𝔽 = set of facts = subset of 𝔼 × ℛ × 𝔼

Then:

```
KB = { (s, r, o) ∈ 𝔼 × ℛ × 𝔼 }
```

#### Logical View

Each triple (s, r, o) is equivalent to a first-order logic assertion:

```
r(s, o)
```

e.g.,

```
bornIn(BarackObama, Honolulu)
```



### Graph Representation

A KB is naturally a **directed labeled multigraph**:

* Nodes = entities
* Edges = labeled by predicates

Example:

```
Barack Obama ──bornIn──▶ Honolulu
```

> We will use libraries like networkx in Python to model this.

In [ ]:
import networkx as nx

G = nx.DiGraph()
G.add_edge("Barack Obama", "Honolulu", relation="bornIn")

: 

In [ ]:
class KnowledgeBase:
    def __init__(self):
        self.triples = set() # A set to store unique triples

    def add_fact(self, subject, predicate, object_):
        self.triples.add((subject, predicate, object_)) # Storing as a tuple

    def query(self, subject=None, predicate=None, object_=None):
        return {
            (s, p, o)
            for s, p, o in self.triples # Iterate through stored triples
            if (subject in [None, s]) and # Check if subject matches or is None
               (predicate in [None, p]) and # Check if predicate matches or is None
               (object_ in [None, o]) # Check if object matches or is None
        }


# Example Usage
kb = KnowledgeBase()
kb.add_fact("Obama", "bornIn", "Honolulu")
kb.add_fact("Obama", "presidentOf", "USA")

# Query all facts where subject is Obama
print(kb.query(subject="Obama"))

### RDF – Resource Description Framework

### Intuition: What is RDF?
RDF is a framework for representing structured data as triples:
```text
(subject, predicate, object)
```
We can think of it as:
* **Subject**: a thing we’re describing
* **Predicate**: a property or relationship
* **Object**: the value or entity connected to the subject

#### Real-World Analogy

Imagine that we have a filing cabinet where every card reads like:

```
(Mona Lisa, paintedBy, Leonardo da Vinci)
(Leonardo da Vinci, bornIn, Vinci)
(Mona Lisa, locatedIn, Louvre)
```

This "card catalog" is the core idea of RDF: **facts represented as triples**.

---

#### Core Concepts and Vocabulary

RDF is part of the **Semantic Web** and uses Internationalized Resource Identifiers(IRIs) instead of simple names.

#### RDF Triple Components

| Component | Role                      | Example                                     |
| --------- | ------------------------- | ------------------------------------------- |
| Subject   | Entity/resource           | `<http://example.org/MonaLisa>`             |
| Predicate | Property/relationship     | `<http://example.org/paintedBy>`            |
| Object    | Value (entity or literal) | `<http://example.org/Leonardo>` or `"1503"` |

---

#### Formal Semantics and Model-Theoretic Foundation

#### RDF Graph

Mathematically, an RDF graph is a **set of triples**:
Let:

* **U** be the set of all URIs (IRIs)
* **L** be the set of all literals (strings, numbers, etc.)
* **B** be a set of blank (anonymous) nodes

Each triple:

```
(s, p, o) ∈ (U ∪ B) × U × (U ∪ B ∪ L)
```

This allows:

* **s** = URI or blank node
* **p** = URI only (predicates must be defined)
* **o** = URI, blank node, or literal

#### Graphical View

Triples form a **directed labeled graph**, with edges = predicates.

```
[Mona Lisa] ──paintedBy──▶ [Leonardo]
[Mona Lisa] ──year──────▶ "1503"
```

### Entailment and Semantics

Given:

```
(:x, :p, :y), (:y, :p, :z)
```

We can define inference rules. For instance, if `p` is transitive:

```
⇒ (:x, :p, :z)
```

This forms the basis of **RDFS** and **OWL** (RDF Schema and Web Ontology Language), extending RDF to support richer logic.

#### Creating RDF In Python

In [ ]:

from rdflib import Graph, URIRef, Literal, Namespace

ex = Namespace("http://example.org/") # Defining a namespace for oour RDF dat
g = Graph()

g.add((ex.MonaLisa, ex.paintedBy, ex.Leonardo)) # Adding a triple to the graph
g.add((ex.MonaLisa, ex.year, Literal("1503"))) # Adding another triple with a literal value

# Print all triples
for s, p, o in g:
    print(s, p, o)

### SPARQL – RDF Query Language

#### Intuition: What is SPARQL?

**SPARQL** is to RDF what **SQL** is to relational databases. It lets us **query RDF graphs** by **matching patterns**.

### Real-World Analogy

In a library, we might ask:

> "Find all paintings created by Leonardo."

In SPARQL:

```sparql
SELECT ?painting
WHERE {
  ?painting <http://example.org/paintedBy> <http://example.org/Leonardo> .
}
```

---

#### SPARQL Query Structure

#### Basic Syntax

```sparql
PREFIX ex: <http://example.org/>

SELECT ?painting ?year
WHERE {
  ?painting ex:paintedBy ex:Leonardo .
  ?painting ex:year ?year .
}
```

* `?var` = query variable
* Each clause in `WHERE` = triple pattern
* Matches data like regex matches strings

#### Filters and Conditions

```sparql
FILTER (?year > "1500")
```

#### Optional Matches

```sparql
OPTIONAL { ?painting ex:location ?loc }
```

#### Aggregates and Grouping

```sparql
SELECT ?artist (COUNT(?painting) AS ?count)
WHERE {
  ?painting ex:paintedBy ?artist .
}
GROUP BY ?artist
```

#### Mathematical View: Pattern Matching

Let:

* G = RDF graph = set of triples
* P = Basic Graph Pattern (BGP) = set of triple patterns with variables

The **evaluation** of BGP `P` over `G` is:

```
eval(P, G) = { μ | dom(μ) = vars(P), and μ(P) ⊆ G }
```

Where:

* `μ` = variable mapping (binding)
* `μ(P)` = set of triples produced by substituting variables

SPARQL answers are derived by evaluating these mappings over the graph.

## 2. Open Information Extraction (Open IE)

### What is Information Extraction (IE)?
Information Extraction (IE) is the task of automatically identifying structured information (like entities and relationships) from unstructured text (like news articles, web pages, etc.). In layman terms it refers to the automatic extraction of structured information from unstructured natural language text. Traditional IE systems typically work with a closed schema: a fixed set of entity types and relation types.<br>
For example, in a medical IE system:
```
Entity types: Drug, Disease, Symptom
Relation types: treats, causes, indicates
```
This type of system might extract information like:
> (Aspirin, treats, Headache) 

Uusally this works well in domain-specific tasks where the schema is known beforehand. But it breaks down when we try to apply it to open-domain corpus of knowledge like the entire World Wide Web.

#### Traditional IE:
- Requires a predefined schema (fixed relations like bornIn, presidentOf).
- Needs domain-specific training data.
- Limited to closed-world assumptions.

#### The Need for Open IE
On the internet there are countless relation types, for example:
- (Barack Obama, wrote, Dreams from My Father)
- (Tesla, acquired, Maxwell Technologies)
- (Python, is popular among, data scientists)

But there is a problem that no predefined schema can capture them all. This is where Open IE comes in.



### Open Information Extraction (Open IE)
Open IE systems take natural language text and extract relational tuples, without requiring a predefined ontology or relation set.<br>
For example, suppose we are given following sentence:
> "Barack Obama was born in Honolulu and later became President of the United States."
Now the Open IE might consider two outputs:
- (Barack Obama, was born in, Honolulu)
- (Barack Obama, became President of, the United States)
The system automatically identifies meaningful triples using general linguistic patterns. The goal is to maintain broad coverage, precision, and scalability.

### Core Concepts and Workflow

#### Step-1: Sentence Segmentation
First of all we break the text into individual sentences.

#### Step-2: Clause Detection
Each sentence is broken into clauses, each representing a minimal factual unit.

> "Barack Obama was born in Hawaii and grew up in Indonesia." →
- Clause 1: "Barack Obama was born in Hawaii"
- Clause 2: "[He] grew up in Indonesia"

#### Step 3: Tuple Extraction

Finally from each clause we extract:
- Subject
- Relation phrase
- Object (or complement)

Which eventually leads us to give results in a triple:
`(subject, relation, object)`


###  Mathematical Foundations

Let the input sentence be a sequence of tokens:

```
S = [w_1, w_2, ..., w_n]
```

Let $T = \{ (a_i, r_i, b_i) \}$ be the set of extracted tuples.

Formally, each triple $(a, r, b)$ must satisfy:

* $a, b \subseteq S$: argument phrases
* $r \subseteq S$: relation phrase

These are subject to syntactic constraints:

* $a$ must be a noun phrase that syntactically acts as a subject
* $b$ must be a noun phrase that syntactically acts as an object
* $r$ is derived from verb phrases and surrounding context

#### Confidence Scoring

A confidence function $f: T \rightarrow [0, 1]$ assigns a score to each extracted triple.

## 3. Named Entity Recognition and Entity Linking

### Why Care About Named Entities?

First of all we start with a question that what does the following words have in common?
> "Einstein," "Google," "Paris," "July 20, 1969," "$1 billion" 

All of these words are named entities i.e. specific, meaningful chunks of language that refer to real-world things: people, organizations, places, dates, monetary values, and more. Machines must understand such entities to answer questions, summarize articles, translate correctly, or even recommend products.
But when it comes to words there arises a problem that <i>words are but wind </i> which means that can be interpreted in many ways and may not always have a fixed or substantial meaning.<br>
For example, take the word “Apple.”
> Are we talking here about a fruit, a tech company, or The Beatles’ record label which was called Apple Records?

This is where Named Entity Recognition (NER) and Entity Linking (EL) come into play — two foundational NLP tasks that convert raw language into structured knowledge.

### What Is Named Entity Recognition?

#### Intuition
Named Entity Recognition is the task of identifying and classifying spans of text that refer to real-world entities.<br>
For example
> “Barack Obama was born in Honolulu in 1961.”

NER output would be:
- “Barack Obama” → `PERSON`
- “Honolulu” → `LOCATION`
- “1961” → `DATE`

This task seems simple for humans. But for machines, it's a complex sequence prediction problem requiring understanding of grammar, semantics, and context.

#### Formal Definition

Let $x = (x_1, x_2, \dots, x_n)$ be a sequence of tokens (words), and $y = (y_1, y_2, \dots, y_n)$ be the corresponding sequence of tags, where each $y_i \in \mathcal{T}$ (tag set like `B-PER`, `I-PER`, `O`, etc.).

The goal of NER is to learn:

$$
f: x \rightarrow y
$$

NER is a **sequence labeling** problem — similar in structure to part-of-speech tagging or shallow parsing.




#### The BIO Tagging Scheme
NER uses structured labels to define which words are part of entities:
- B-<TYPE>: Beginning of an entity
- I-<TYPE>: Inside of an entity
- O: Outside of any entity

##### For Example: 

| Token    | Tag    |
| -------- | ------ |
| Barack   | B-PER  |
| Obama    | I-PER  |
| was      | O      |
| born     | O      |
| in       | O      |
| Honolulu | B-LOC  |
| in       | O      |
| 1961     | B-DATE |

This tagging structure enables models to recognize multi-word named entities with consistent boundaries. 

###  How is NER Solved?

#### 1. Rule-based (Early Days)

Early systems used regular expressions and hand-crafted rules:

* Capitalization
* Trigger words (“President”, “Inc.”)
* Gazetteers (lists of known names)

**Limitation**: brittle, not generalizable.


#### 2. Statistical Models

##### Hidden Markov Models (HMM)

* Probabilistic models that treat NER as a generative sequence process.
* Lacked flexibility in features.

##### Conditional Random Fields (CRFs)

CRFs model the conditional probability:

$$
P(y | x) = \frac{1}{Z(x)} \exp\left( \sum_{i=1}^{n} \psi(y_i, y_{i-1}, x, i) \right)
$$

Where:

* $\psi$ is a learned potential function capturing feature dependencies.
* $Z(x)$ is the normalization constant.

They capture dependencies across tags — e.g., "I-PER" cannot follow "B-LOC".


#### 3. Neural Approaches

##### BiLSTM-CRF

* Bidirectional LSTM extracts context-aware embeddings.
* CRF layer models label dependencies.
* Represented a huge leap in accuracy.

##### Transformer-Based NER (e.g., BERT)

* Use pretrained language models like **BERT**, **RoBERTa**, or **DeBERTa**.
* Fine-tune them for sequence classification.

**Architecture:**

* Input: tokenized sentence
* Output: label for each token

Let $h_i$ be the contextualized embedding from BERT for token $x_i$. Then:

$$
P(y_i | x) = \text{Softmax}(W h_i + b)
$$

